In [1]:
import math
import os
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO))

os.environ["DISPLAY"] = ":0"
# os.environ["ISAAC_RENDER"] = "0"

# No local Isaac window; watch the viewport over WebRTC instead (port 49100).
# os.environ["ISAAC_HEADLESS"] = "1"
# os.environ["ISAAC_LIVESTREAM"] = "1"

# os.environ["ISAAC_WINDOW"] = "1280x720"
# os.environ["ISAAC_WINDOW"] = "960x540"
# os.environ["ISAAC_WINDOW"] = "854x480"
# os.environ["ISAAC_WINDOW"] = "768x432"
os.environ["ISAAC_WINDOW"] = "640x360"
# os.environ["ISAAC_WINDOW"] = "512x288"

from launcher import (
    start_giskard_server,
    start_isaac_sim,
    start_rviz,
    start_streaming_client,
    stop,
)

from cram_vrb_lab.sim.isaac_app import livestream_enabled

RVIZ_CONFIG = REPO / "demos" / "rviz" / "garmi.rviz"
ROBOT, SCENE = "garmi", "garmi_apartment"


SPAWN_POSITION = (0, 5.0, 0.0259)
SPAWN_YAW = -math.pi / 2

# rviz_proc = start_rviz(rviz_config=RVIZ_CONFIG)
# sim_proc = start_isaac_sim(robot=ROBOT, scene=SCENE, camera="none",
#                            spawn_position=SPAWN_POSITION, spawn_yaw=SPAWN_YAW)
# stream_proc = start_streaming_client() if livestream_enabled() else None
# giskard_proc = start_giskard_server(robot=ROBOT, scene=SCENE,
#                                     spawn_position=SPAWN_POSITION, spawn_yaw=SPAWN_YAW)

# from time import sleep
# while True:
#     sleep(10)
# %%
import threading

import nest_asyncio
import numpy as np
import rclpy
from rclpy.executors import MultiThreadedExecutor

nest_asyncio.apply()

from coraplex.datastructures.dataclasses import Context
from semantic_digital_twin.adapters.ros.world_fetcher import fetch_world_from_service
from semantic_digital_twin.adapters.ros.world_synchronizer import WorldSynchronizer

from cram_vrb_lab.robots.garmi.motions import GARMI_MOTION_MAPPINGS
from semantic_digital_twin.robots.garmi import Garmi

if not rclpy.ok():
    rclpy.init()
node = rclpy.create_node('cram_garmi_node')
executor = MultiThreadedExecutor()
executor.add_node(node)
threading.Thread(target=executor.spin, daemon=True, name='rclpy-executor').start()

world = fetch_world_from_service(node=node, timeout_seconds=300)
WorldSynchronizer(_world=world, node=node)

robot = world.get_semantic_annotations_by_type(Garmi)
robot = robot[0] if robot else Garmi.from_world(world)

robot.mobile_base.full_body_controlled = True

context = Context(
    world=world,
    robot=robot,
    ros_node=node,
    evaluate_conditions=False,
    alternative_motion_mappings=GARMI_MOTION_MAPPINGS,
)
print('connected, robot:', type(robot).__name__)
print('bodies in the twin:', len(world.bodies), '-- GARMI plus the whole flat')

# %%
from coraplex.datastructures.enums import Arms
from coraplex.execution_environment import real_robot, simulated_robot
from coraplex.plans.factories import execute_single, sequential
from coraplex.view_manager import ViewManager
from giskardpy.data_types.exceptions import GiskardException


def run_plan(plan, collision_avoidance=True, strict=False):
    try:
        with real_robot(collision_avoidance=collision_avoidance):
            plan.perform()
    except GiskardException as failure:
        if strict:
            raise
        # str() on these carries giskard's own error_message() and, where it has
        # one, its suggested correction.
        print(f'giskard failed -- {type(failure).__name__}: {failure}')
        return False
    print('done')
    return True


def tool_frame_matrix(arm=Arms.LEFT):
    """The tool frame's 4x4 pose in map."""
    tool = ViewManager.get_end_effector_view(arm, robot).tool_frame
    return np.asarray(tool.global_pose.to_np())


def tool_position(arm=Arms.LEFT):
    return tool_frame_matrix(arm)[:3, 3].ravel()


def tool_axis(arm=Arms.LEFT):
    return tool_frame_matrix(arm)[:3, 0].ravel()


def closing_axis(arm=Arms.LEFT):
    return tool_frame_matrix(arm)[:3, 1].ravel()


def body_position(name):
    return np.asarray(world.get_body_by_name(name).global_pose.to_np())[:3, 3].ravel()

# %%
from coraplex.robot_plans.actions.core.robot_body import ParkArmsAction, SetGripperAction
from semantic_digital_twin.datastructures.definitions import GripperState

from coraplex.robot_plans.actions.core.navigation import NavigateAction
from semantic_digital_twin.spatial_types import Point3, Quaternion
from semantic_digital_twin.spatial_types.spatial_types import Pose

STANDOFF = 1.6
def station_facing(handle_x):
    return Pose(
        Point3.from_iterable([handle_x - 0.3, 7.12 - STANDOFF, 0.0]),
        Quaternion.from_iterable([0.0, 0.0, math.sin(math.pi / 4), math.cos(math.pi / 4)]),
        reference_frame=world.root,
    )


ARRIVED = 0.05
"""How close to the station counts as arrived [m]."""


def drive_to(handle_name, attempts=10):
    target = station_facing(float(body_position(handle_name)[0]))
    goal = np.asarray(target.to_np())[:2, 3].ravel()
    for attempt in range(1, attempts + 1):
        run_plan(execute_single(NavigateAction(target), context=context))
        error = float(np.linalg.norm(body_position('base_link')[:2] - goal))
        print(f'  navigate {attempt}: base {np.round(body_position("base_link"), 3)}'
              f' error {error:.3f} m')
        if error <= ARRIVED:
            break
    else:
        print(f'  WARNING: still {error:.3f} m from the station after {attempts} tries')
    print('at', handle_name, np.round(body_position(handle_name), 3))

from coraplex.datastructures.enums import ApproachDirection, VerticalAlignment
from coraplex.datastructures.grasp import GraspDescription
from coraplex.robot_plans.actions.core.pick_up import GraspingAction
from coraplex.robot_plans.motions.container import ClosingMotion, OpeningMotion
from coraplex.robot_plans.motions.gripper import MoveGripperMotion
from semantic_digital_twin.semantic_annotations.semantic_annotations import (
    Drawer,
    Handle,
    Door,
)

from coraplex.robot_plans.actions.core.container import OpenAction, CloseAction

GRASPED = 0.01
"""How close the tool frame has to land to the commanded grasp pose [m]."""


def _grasp_description(arm):
    """The grasp OpenAction uses internally: approach the handle head on."""
    return GraspDescription(
        ApproachDirection.FRONT,
        VerticalAlignment.NoAlignment,
        ViewManager.get_end_effector_view(arm, robot),
    )


def report_grasp_geometry(handle_body, arm=Arms.LEFT):
    box = handle_body.collision.as_bounding_box_collection_in_frame(
        handle_body
    ).bounding_box()
    # min_*/max_* are relative to the box's own origin, not to the body frame.
    centre = np.array([
        float(box.origin.x) + (box.min_x + box.max_x) / 2,
        float(box.origin.y) + (box.min_y + box.max_y) / 2,
        float(box.origin.z) + (box.min_z + box.max_z) / 2,
    ])
    print(f'  handle origin (grasp target): {np.round(body_position(handle_body.name.name), 4)}')
    print(f'  collision box in body frame : centre {np.round(centre, 4)}'
          f' size {np.round(np.array(box.dimensions), 4)}')
    print(f'  -> CRAM aims {np.linalg.norm(centre) * 1000:.1f} mm off the collision centre')


def grasp_handle(handle_body, arm=Arms.LEFT, attempts=6):
    grasp = _grasp_description(arm)
    _, commanded, _ = grasp.grasp_pose_sequence(handle_body)
    # grasp_pose_sequence works in the *handle's* frame (it starts from
    # Pose(reference_frame=body)), so it has to be lifted into map before it can
    # be compared with where the tool actually is.
    goal_frame = (np.asarray(handle_body.global_pose.to_np())
                  @ np.asarray(commanded.to_np()))
    goal = goal_frame[:3, 3].ravel()

    for attempt in range(1, attempts + 1):
        run_plan(execute_single(GraspingAction(handle_body, arm, grasp),
                                context=context),
                 collision_avoidance=True)
        residual = tool_position(arm) - goal
        # Resolved along the axes of the pose that was *asked* for, so the three
        # numbers keep meaning the same thing however the arm ended up oriented.
        approach, closing, lift = (
            float(residual @ goal_frame[:3, i]) for i in range(3)
        )
        error = float(np.linalg.norm(residual))
        print(f'  grasp {attempt}: residual {error * 1000:6.1f} mm'
              f'  (approach {approach * 1000:+6.1f}, closing {closing * 1000:+6.1f},'
              f' lift {lift * 1000:+6.1f} mm)')
        if error <= GRASPED:
            return True
    print(f'  WARNING: tool still {error * 1000:.1f} mm off after {attempts} tries')
    return False


def _work_container(motion, handle_body, arm, attempts):
    report_grasp_geometry(handle_body, arm)
    grasp_handle(handle_body, arm, attempts)
    run_plan(execute_single(motion(handle_body, arm), context=context),
             collision_avoidance=True)
    run_plan(execute_single(MoveGripperMotion(GripperState.OPEN, arm), context=context),
             collision_avoidance=True)


def open_container(handle_body, arm=Arms.LEFT, attempts=3):
    """OpenAction, with the reach measured and retried. See the note above."""
    _work_container(OpeningMotion, handle_body, arm, attempts)


def close_container(handle_body, arm=Arms.LEFT, attempts=3):
    """CloseAction, likewise -- it is the same three steps with ClosingMotion."""
    _work_container(ClosingMotion, handle_body, arm, attempts)


connected, robot: Garmi
bodies in the twin: 145 -- GARMI plus the whole flat


In [8]:
STANDOFF = 1.3
STANDOFF_L = 0
robot.mobile_base.full_body_controlled = True

def station_facing(handle_x):
    return Pose(
        Point3.from_iterable([handle_x + STANDOFF_L, 7.12 - STANDOFF, 0.0]),
        Quaternion.from_iterable([0.0, 0.0, math.sin(math.pi / 4), math.cos(math.pi / 4)]),
        reference_frame=world.root,
    )

In [ ]:
STANDOFF = 1.2
robot.mobile_base.full_body_controlled = False

In [9]:
def reset_pos():
    drive_to(f"cabinet_door_1_handle")
    run_plan(sequential([
            SetGripperAction(Arms.LEFT, GripperState.OPEN),
            SetGripperAction(Arms.RIGHT, GripperState.OPEN),
            ParkArmsAction(Arms.LEFT),
            ParkArmsAction(Arms.RIGHT),
        ], context=context))

reset_pos()

[INFO] [1786963603.741946552] [cram_garmi_node]: giskard/command Goal #0 accepted
[INFO] [1786963604.770442078] [cram_garmi_node]: giskard/command Goal #0 result received


done
  navigate 1: base [0.672 5.82  0.   ] error 0.001 m
at cabinet_door_1_handle [0.673 7.116 0.776]


[INFO] [1786963605.181338388] [cram_garmi_node]: giskard/command Goal #0 accepted


done


[INFO] [1786963612.771284279] [cram_garmi_node]: giskard/command Goal #0 result received


In [7]:
ROUNDS = 1
"""How many times to work the three drawers, for a long unattended run."""

for round_id in range(1, ROUNDS + 1):
    print(f"===== round {round_id}/{ROUNDS} =====")
    for drawer_id in range(1, 5):
        _arm = Arms.RIGHT if drawer_id == 4 else Arms.LEFT
        _arm = Arms.RIGHT
        drawer_body = world.get_body_by_name(f"drawer_{drawer_id}")
        handle_body = world.get_body_by_name(f"drawer_{drawer_id}_handle")

        if not world.get_semantic_annotations_by_type(Drawer):
            with world.modify_world():
                world.add_semantic_annotation_recursively(
                    Drawer(root=drawer_body, handle=Handle(root=handle_body))
                )
        print("drawer annotated:", drawer_body.name, "with handle", handle_body.name)
        print("handle at", np.round(body_position(f"drawer_{drawer_id}_handle"), 3))

        drive_to(f"drawer_{drawer_id}_handle")
        open_container(handle_body, _arm)
        print("Opened: drawer joint:", world.get_connection_by_name(f"drawer_{drawer_id}_joint").position)

        close_container(handle_body, _arm)
        print("Closed: drawer joint:", world.get_connection_by_name(f"drawer_{drawer_id}_joint").position)

    drive_to(f"drawer_{drawer_id}_handle")
    run_plan(sequential([
        ParkArmsAction(Arms.LEFT),
        SetGripperAction(Arms.LEFT, GripperState.OPEN),
        ParkArmsAction(Arms.RIGHT),
        SetGripperAction(Arms.RIGHT, GripperState.OPEN),
    ], context=context))


===== round 1/1 =====
drawer annotated: drawer_1 with handle drawer_1_handle
handle at [-0.09   7.123  0.8  ]


[INFO] [1786963539.784197521] [cram_garmi_node]: giskard/command Goal #0 accepted
[INFO] [1786963545.509420934] [cram_garmi_node]: giskard/command Goal #0 result received


done
  navigate 1: base [-0.108  5.82   0.   ] error 0.018 m
at drawer_1_handle [-0.09   7.123  0.8  ]


[INFO] [1786963545.921463288] [cram_garmi_node]: giskard/command Goal #0 accepted
[INFO] [1786963550.960814541] [cram_garmi_node]: giskard/command Goal #0 result received


done
  navigate 1: base [0.69 5.82 0.  ] error 0.018 m
at cabinet_door_1_handle [0.673 7.116 0.776]


...

[INFO] [1786963582.855595261] [cram_garmi_node]: giskard/command Goal #0 accepted
[INFO] [1786963588.868312937] [cram_garmi_node]: giskard/command Goal #0 result received


done
  navigate 1: base [-0.09  5.82  0.  ] error 0.000 m
at drawer_2_handle [-0.09   7.123  0.6  ]


[INFO] [1786963589.309154834] [cram_garmi_node]: giskard/command Goal #0 accepted


KeyboardInterrupt: 

[INFO] [1786963594.318939960] [cram_garmi_node]: giskard/command Goal #0 result received


In [ ]:
door_id = "1"
door_body = world.get_body_by_name(f"cabinet_door_{door_id}")
door_handle_body = world.get_body_by_name(f"cabinet_door_{door_id}_handle")

if not world.get_semantic_annotations_by_type(Door):
    with world.modify_world():
        world.add_semantic_annotation_recursively(
            Door(root=door_body, handle=Handle(root=door_handle_body))
        )
print("Door annotated:", door_body.name, "with handle", door_handle_body.name)

run_plan(sequential([
        ParkArmsAction(Arms.LEFT),
        ParkArmsAction(Arms.RIGHT),
        SetGripperAction(Arms.LEFT, GripperState.OPEN),
        SetGripperAction(Arms.RIGHT, GripperState.OPEN),
    ], context=context))

_arm = Arms.RIGHT

drive_to(f"cabinet_door_{door_id}_handle")
open_container(door_handle_body, _arm)
print("Opened, Door joint:", world.get_connection_by_name(f"cabinet_door_{door_id}_joint").position)

close_container(door_handle_body, _arm)
print("Closed, Door joint:", world.get_connection_by_name(f"cabinet_door_{door_id}_joint").position)

drive_to(f"cabinet_door_{door_id}_handle")

run_plan(sequential([
    ParkArmsAction(Arms.LEFT),
    ParkArmsAction(Arms.RIGHT),
    SetGripperAction(Arms.LEFT, GripperState.OPEN),
    SetGripperAction(Arms.RIGHT, GripperState.OPEN),
], context=context))

Door annotated: cabinet_door_1 with handle cabinet_door_1_handle


[INFO] [1786963302.707043810] [cram_garmi_node]: giskard/command Goal #0 accepted
[INFO] [1786963313.749536383] [cram_garmi_node]: giskard/command Goal #0 result received


done


[INFO] [1786963314.116498011] [cram_garmi_node]: giskard/command Goal #0 accepted
[INFO] [1786963331.500336337] [cram_garmi_node]: giskard/command Goal #0 result received


done
  navigate 1: base [0.172 5.62  0.   ] error 0.000 m
at cabinet_door_1_handle [0.672 7.121 0.776]
  handle origin (grasp target): [0.6724 7.1208 0.776 ]
  collision box in body frame : centre [-0.0154  0.      0.    ] size [0.0118 0.0118 0.092 ]
  -> CRAM aims 15.4 mm off the collision centre


...

[INFO] [1786963392.374973953] [cram_garmi_node]: giskard/command Goal #0 accepted
[INFO] [1786963393.552777532] [cram_garmi_node]: giskard/command Goal #0 result received


done
Closed, Door joint: 0.010702594942103088


[INFO] [1786963393.897163919] [cram_garmi_node]: giskard/command Goal #0 accepted


done
  navigate 1: base [0.173 5.6   0.   ] error 0.020 m
at cabinet_door_1_handle [0.673 7.116 0.776]


[INFO] [1786963398.051446892] [cram_garmi_node]: giskard/command Goal #0 result received
